# Silver layer

Creation of the silver layer based on the following query proven to be incrementalisable

In [0]:
CREATE MATERIALIZED VIEW language_app.silver.lesson_attempts_enriched
SELECT
    -- -------------------------------------------------------------------------
    -- Attempt (event) columns
    -- Defensive typing: CSV inference in bronze is best-effort, so silver
    -- casts explicitly to guarantee the types downstream consumers rely on.
    -- No COALESCE needed: these come from the driving table, and bronze
    -- expectations already dropped rows with missing IDs or invalid scores.
    -- -------------------------------------------------------------------------
      CAST(rla.attempt_id AS BIGINT)                    AS attempt_id
    , CAST(rla.learner_id AS INT)                       AS learner_id        -- kept from the attempt (not rl) so unknown IDs stay visible
    , rla.course_id
    , CAST(rla.lesson_number AS INT)                    AS lesson_number
    , CAST(rla.attempted_at AS TIMESTAMP)               AS attempted_at
    , CAST(rla.attempted_at AS DATE)                    AS attempt_date      -- derived once here so gold daily aggregates don't repeat it
    , CAST(rla.score AS INT)                            AS score
    , CASE WHEN rla.minutes_spent >= 0
           THEN CAST(rla.minutes_spent AS INT)
      END                                               AS minutes_spent     -- negatives were WARNed (kept) in bronze; NULL = "unknown",
                                                                             -- which AVG() skips. 0 would falsely drag averages down.
    , CAST(rla.completed AS BOOLEAN)                    AS completed
    , rla.device

    -- -------------------------------------------------------------------------
    -- Learner columns (from source.raw_learners via LEFT JOIN)
    -- Unmatched attempts get NULLs here; apply the NULL-handling rule.
    -- -------------------------------------------------------------------------
    , rl.learner_id IS NOT NULL                         AS learner_found     -- false = attempt references a learner that doesn't exist
    , COALESCE(rl.first_name || ' ' || rl.last_name,
               'Unknown')                               AS learner_name      -- text dimension -> 'Unknown'
    , COALESCE(rl.country, 'Unknown')                   AS learner_country   -- text dimension -> 'Unknown'
    , COALESCE(rl.native_language, 'Unknown')           AS native_language   -- text dimension -> 'Unknown'
    , COALESCE(rl.subscription_tier, 'Unknown')         AS subscription_tier -- text dimension -> 'Unknown'
    , CAST(rl.signup_date AS DATE)                      AS signup_date       -- date -> stays NULL (a fake date would break date maths)

    -- -------------------------------------------------------------------------
    -- Course columns (from bronze.raw_courses via LEFT JOIN)
    -- All course_ids match today; the rule still applies in case of
    -- late-arriving courses or bad keys in future batches.
    -- -------------------------------------------------------------------------
    , COALESCE(rc.language, 'Unknown')                  AS course_language   -- text dimension -> 'Unknown'
    , COALESCE(rc.level, 'Unknown')                     AS course_level      -- text dimension -> 'Unknown'
    , COALESCE(rc.course_name, 'Unknown')               AS course_name       -- text dimension -> 'Unknown'
    , CAST(rc.total_lessons AS INT)                     AS course_total_lessons -- number -> stays NULL (0 would cause divide-by-zero)

FROM language_app.bronze.raw_lesson_attempts rla

-- LEFT JOINs throughout: silver flags problems rather than silently dropping rows.
-- An INNER JOIN would quietly lose attempts with unknown learners/courses.

-- Learners are joined directly from source as a simplification;
-- in production this would come through CDC (Module 3).

LEFT JOIN language_app.source.raw_learners rl
    ON rla.learner_id = rl.learner_id
LEFT JOIN language_app.bronze.raw_courses rc
    ON rla.course_id = rc.course_id
;